# Estudo de Caso Guiado — Onde está concentrada a manutenção?

**Projeto:** `psf/requests`

**Objetivo:** usar o histórico Git para identificar arquivos que concentram manutenção e explorar os principais temas das mensagens de commit.

## Perguntas de investigação

- **RQ1:** Quais arquivos foram modificados com maior frequência?
- **RQ2:** Quais arquivos apresentam maior churn de código?
- **RQ3:** Quais são os principais temas encontrados nas mensagens de commit?

## Fluxo da análise

GitHub → PyDriller → Pandas → Visualização → Topic Modelling → Interpretação

## 1. Baixar o projeto

Para manter a análise rápida durante a oficina, vamos usar apenas os **500 commits mais recentes**.

> Essa escolha já é uma decisão metodológica: estamos analisando uma janela do histórico, não todo o projeto.

In [ ]:
from pathlib import Path
import subprocess

URL = "https://github.com/psf/requests.git"
REPO = Path("/workspace/repositories/requests")

REPO.parent.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "500", URL, str(REPO)],
        check=True
    )

print("Repositório disponível em:", REPO)

## 2. Minerar commits com PyDriller

Vamos transformar o histórico Git em uma tabela.

Cada linha do nosso dataset representará **um arquivo modificado em um commit**.

In [ ]:
from pydriller import Repository

dados = []

for commit in Repository(str(REPO)).traverse_commits():

    for arquivo in commit.modified_files:

        caminho = arquivo.new_path or arquivo.old_path

        dados.append({
            "commit": commit.hash,
            "data": commit.author_date,
            "autor": commit.author.name,
            "mensagem": commit.msg.splitlines()[0],
            "arquivo": caminho,
            "adicionadas": arquivo.added_lines,
            "removidas": arquivo.deleted_lines
        })

print("Registros coletados:", len(dados))

## 3. Transformar os dados em DataFrame

Um commit pode aparecer várias vezes porque um mesmo commit pode modificar vários arquivos.

In [ ]:
import pandas as pd

df = pd.DataFrame(dados)

df.head()

### Inspeção rápida

Antes de analisar qualquer coisa, vamos entender o tamanho do dataset.

In [ ]:
print("Modificações de arquivo:", len(df))
print("Commits únicos:", df["commit"].nunique())
print("Autores únicos:", df["autor"].nunique())
print("Arquivos únicos:", df["arquivo"].nunique())

In [ ]:
print("Autores com mais modificações registradas:")
display(df["autor"].value_counts().head(10))

# RQ1 — Quais arquivos mudam mais?

Vamos contar quantas vezes cada arquivo aparece no histórico analisado.

> Um arquivo que muda frequentemente pode ser um **change hotspot**, mas isso não significa automaticamente que ele tenha baixa qualidade.

In [ ]:
frequencia = (
    df.groupby("arquivo")
      .size()
      .reset_index(name="modificacoes")
      .sort_values("modificacoes", ascending=False)
)

frequencia.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

top10 = frequencia.head(10)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top10,
    x="modificacoes",
    y="arquivo"
)
plt.title("Arquivos modificados com maior frequência")
plt.xlabel("Número de modificações")
plt.ylabel("Arquivo")
plt.show()

## Interpretação

Perguntas para discutir com a turma:

- Os arquivos mais modificados parecem ser centrais ao projeto?
- Há arquivos de código, testes ou documentação entre os mais modificados?
- Um arquivo muito modificado é necessariamente problemático?

# RQ2 — Quais arquivos apresentam maior churn?

Vamos definir uma medida simples:

**churn = linhas adicionadas + linhas removidas**

In [ ]:
df["churn"] = df["adicionadas"] + df["removidas"]

df[["arquivo", "adicionadas", "removidas", "churn"]].head()

In [ ]:
churn = (
    df.groupby("arquivo")["churn"]
      .sum()
      .reset_index()
      .sort_values("churn", ascending=False)
)

churn.head(10)

In [ ]:
top_churn = churn.head(10)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_churn,
    x="churn",
    y="arquivo"
)
plt.title("Arquivos com maior churn")
plt.xlabel("Churn total")
plt.ylabel("Arquivo")
plt.show()

## Frequência × Churn

Agora vamos combinar as duas evidências.

In [ ]:
hotspots = frequencia.merge(
    churn,
    on="arquivo"
)

hotspots.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=hotspots,
    x="modificacoes",
    y="churn"
)
plt.title("Frequência de modificação × Churn")
plt.xlabel("Número de modificações")
plt.ylabel("Churn total")
plt.show()

## Interpretação

Arquivos com:

- muitas modificações;
- e muito churn;

podem ser bons candidatos para uma investigação mais detalhada.

> **Hotspot não é sinônimo de defeito.** É uma evidência que ajuda a priorizar onde investigar.

# RQ3 — Quais temas aparecem nas mensagens de commit?

Agora vamos explorar o conteúdo textual das mensagens de commit.

Primeiro, queremos apenas **uma mensagem por commit**.

In [ ]:
commits = (
    df[["commit", "mensagem"]]
    .drop_duplicates()
)

print("Mensagens de commit:", len(commits))
commits.head()

## Remover merges

Mensagens como `Merge branch...` ou `Merge pull request...` normalmente dizem pouco sobre a natureza da mudança.

In [ ]:
commits = commits[
    ~commits["mensagem"]
      .str.lower()
      .str.startswith("merge")
].copy()

print("Mensagens após remover merges:", len(commits))

## Representar texto numericamente com TF-IDF

O algoritmo de topic modelling não trabalha diretamente com texto.

Precisamos transformar as mensagens em uma representação numérica.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.9
)

X = vectorizer.fit_transform(
    commits["mensagem"]
)

print("Número de mensagens:", X.shape[0])
print("Número de termos:", X.shape[1])

## Topic modelling com NMF

Vamos pedir ao algoritmo que encontre **4 tópicos**.

Esse número é uma escolha do pesquisador e pode ser alterado.

In [ ]:
from sklearn.decomposition import NMF

modelo = NMF(
    n_components=4,
    random_state=42
)

modelo.fit(X)

## Mostrar as palavras mais importantes de cada tópico

O algoritmo não dá um nome ao tópico.

Ele retorna grupos de palavras; a interpretação é humana.

In [ ]:
palavras = vectorizer.get_feature_names_out()

for i, topico in enumerate(modelo.components_):

    indices = topico.argsort()[-8:][::-1]

    principais = [
        palavras[j]
        for j in indices
    ]

    print(
        f"Tópico {i + 1}:",
        ", ".join(principais)
    )

## Interpretação dos tópicos

Observe os grupos de palavras e tente atribuir um significado.

Exemplos hipotéticos:

- `docs, documentation, readme, example` → documentação
- `test, regression, assert, fix` → testes/correções
- `dependency, version, package, update` → dependências

> Não trate o nome do tópico como uma verdade objetiva. É uma interpretação baseada nas palavras encontradas.

# Conclusões

Voltando às perguntas:

### RQ1
Quais arquivos foram modificados com maior frequência?

### RQ2
Quais arquivos apresentaram maior churn?

### RQ3
Quais tipos de atividade parecem aparecer nas mensagens de commit?

## Limitações

Antes de concluir, lembre-se:

- analisamos apenas uma janela do histórico;
- frequência de mudança não significa baixa qualidade;
- churn não é uma medida direta de defeitos;
- mensagens de commit podem ser curtas ou pouco informativas;
- topic modelling depende das escolhas de pré-processamento e do número de tópicos.

## Principal mensagem da aula

**MSR é o processo de transformar rastros de desenvolvimento em evidências para responder perguntas de Engenharia de Software.**